# CottonLens â€” full-year price/risk workbench

Run All defaults to status. Frozen historical experiments are preserved. Small CPU pilot uses the same ledger and no GPU environment. Select one workflow below; completed fits are reused. Large searches and export stay closed until pilot decisions and release checks.


`agri-nonlinear-pilot-v1`: one fixed Cotton-only vs shared XGBoost GPU comparison. Select pilot deliberately; no parameter search. Status/compare remain CPU/read-only.


In [ ]:
WORKFLOW = 'status'  # status, review, prepare, pilot_plan, pilot, compare
MAX_MINUTES = 60.
RUN_TRAINING = False
WORKFLOWS = {'status':[('status','A')], 'review':[('review','A')], 'prepare':[('prepare','A')], 'pilot_plan':[('pilot-plan','A')], 'pilot':[('pilot','A')], 'compare':[('compare','A')]}
assert WORKFLOW in WORKFLOWS
STAGES = WORKFLOWS[WORKFLOW]
REFERENCE_EXPERIMENT = 'research-market-v1'
PROFILE = 'full-year-v1'  # Or ams-exploration-v1 / fas-exploration-v1 / nass-exploration-v1 / wasde-exploration-v1 / cftc-exploration-v1 / fx-exploration-v1 / crop-exploration-v1 / weather-exploration-v1 / oncall-exploration-v1 (Tier A)
assert PROFILE in ('full-year-v1','ams-exploration-v1','fas-exploration-v1','nass-exploration-v1','wasde-exploration-v1','cftc-exploration-v1','fx-exploration-v1','crop-exploration-v1','weather-exploration-v1','oncall-exploration-v1','oncall-exploration-v2','recency-pilot-v1','return-path-pilot-v1','agri-transfer-pilot-v1','agri-nonlinear-pilot-v1')
EXPERIMENT = 'research-full-year-v1-r2'
if PROFILE == 'ams-exploration-v1': EXPERIMENT = 'research-ams-exploration-v1'
if PROFILE == 'fas-exploration-v1': EXPERIMENT = 'research-fas-exploration-v1'
if PROFILE == 'nass-exploration-v1': EXPERIMENT = 'research-nass-exploration-v1'
if PROFILE == 'wasde-exploration-v1': EXPERIMENT = 'research-wasde-exploration-v1'
if PROFILE == 'cftc-exploration-v1': EXPERIMENT = 'research-cftc-exploration-v1'
if PROFILE == 'fx-exploration-v1': EXPERIMENT = 'research-fx-exploration-v1'
if PROFILE == 'crop-exploration-v1': EXPERIMENT = 'research-crop-exploration-v1'
if PROFILE == 'weather-exploration-v1': EXPERIMENT = 'research-weather-exploration-v1'
if PROFILE == 'oncall-exploration-v1': EXPERIMENT = 'research-oncall-exploration-v1'
if PROFILE == 'oncall-exploration-v2': EXPERIMENT = 'research-oncall-exploration-v2'
if PROFILE == 'recency-pilot-v1': EXPERIMENT = 'research-recency-pilot-v1'
if PROFILE == 'return-path-pilot-v1': EXPERIMENT = 'research-return-path-pilot-v1'
if PROFILE == 'agri-transfer-pilot-v1': EXPERIMENT = 'research-agri-transfer-pilot-v1'
if PROFILE == 'agri-nonlinear-pilot-v1': EXPERIMENT = 'research-agri-nonlinear-pilot-v1'

# Reporting fixes use current code; resumed fits use their original frozen source.
SOURCE_BUNDLE = 'source-agri-transfer-pilot-v1-20261003.zip'  # Reporting-only percentage correction; frozen fit sources below.
if PROFILE == 'agri-nonlinear-pilot-v1': SOURCE_BUNDLE = 'source-agri-nonlinear-pilot-v1-20261003.zip'
if WORKFLOW in ('prepare','pilot_plan','pilot'):
    SOURCE_BUNDLE = ('source-agri-nonlinear-pilot-v1-20261003.zip' if PROFILE == 'agri-nonlinear-pilot-v1'
                     else 'source-agri-transfer-pilot-v1-20261003.zip' if PROFILE == 'agri-transfer-pilot-v1'
                     else 'source-return-path-pilot-v1-20261003.zip' if PROFILE == 'return-path-pilot-v1'
                     else 'source-recency-pilot-v1-20261003.zip' if PROFILE == 'recency-pilot-v1'
                     else 'source-oncall-exploration-v3-20261003.zip' if PROFILE == 'oncall-exploration-v2'
                     else 'source-oncall-exploration-v1-20261002.zip' if PROFILE == 'oncall-exploration-v1'
                     else 'source-weather-exploration-v1-20261002.zip' if PROFILE == 'weather-exploration-v1'
                     else 'source-crop-exploration-v1-20261002.zip' if PROFILE == 'crop-exploration-v1'
                     else 'source-ams-exploration-v1-20261002.zip' if PROFILE == 'ams-exploration-v1'
                     else 'source-fas-exploration-v1-20261002.zip' if PROFILE == 'fas-exploration-v1'
                     else 'source-nass-exploration-v1-20261002.zip' if PROFILE == 'nass-exploration-v1'
                     else 'source-wasde-exploration-v1-20261002.zip' if PROFILE == 'wasde-exploration-v1'
                     else 'source-cftc-exploration-v1-20261002.zip' if PROFILE == 'cftc-exploration-v1'
                     else 'source-fx-exploration-v1-20261002.zip' if PROFILE == 'fx-exploration-v1'
                     else 'source-full-year-v1-r2-20261001.zip')
assert not (PROFILE in ('ams-exploration-v1','fas-exploration-v1','nass-exploration-v1','wasde-exploration-v1','cftc-exploration-v1','fx-exploration-v1','crop-exploration-v1','weather-exploration-v1','oncall-exploration-v1','oncall-exploration-v2') and WORKFLOW == 'review'), 'Use compare for information experiments'

assert not (PROFILE == 'recency-pilot-v1' and WORKFLOW == 'review'), 'Use compare for recency'

assert not (PROFILE == 'return-path-pilot-v1' and WORKFLOW == 'review'), 'Use compare for return path'

assert not (PROFILE == 'agri-nonlinear-pilot-v1' and WORKFLOW == 'review'), 'Use compare for nonlinear transfer'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path, PurePosixPath
import hashlib, importlib, json, sys, zipfile
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
SOURCE_ZIP = DRIVE_ROOT / 'bundles' / SOURCE_BUNDLE
EXPECTED_ZIP_SHA256 = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert expected_zip == EXPECTED_ZIP_SHA256, 'Source ZIP sidecar differs from pinned bundle'
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA256, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'], 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup_cpu, setup, run
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
import colab_setup
colab_setup.UV_VERSION = '0.12.14'  # Newer interpreter catalog includes the pinned Python 3.12.14.
GPU_FIT = PROFILE == 'agri-nonlinear-pilot-v1' and WORKFLOW in ('prepare','pilot')
PYTHON = setup(research=True) if GPU_FIT else setup_cpu()  # Locked separate environments.
from colab_setup import process_env
def cpu_env():
    env = process_env()
    env.update({'PYTHONPATH':str(REPO/'ml/src'), 'OMP_NUM_THREADS':'2', 'OPENBLAS_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'COTTONLENS_ALLOW_LOCAL_CPU_TABULAR':'1'})
    if GPU_FIT: env.pop('COTTONLENS_ALLOW_LOCAL_CPU_TABULAR',None)
    return env
if GPU_FIT:
    run([PYTHON,'-c','from cottonlens_ml.research.models import gpu_sample; print(gpu_sample())'],env=cpu_env())
reference = DRIVE_ROOT/'experiments'/REFERENCE_EXPERIMENT
EVIDENCE_ROOT = DRIVE_ROOT/'reports/research-readiness-evidence-v1'
READINESS = EVIDENCE_ROOT/'output/reports/free-source-readiness-20261001-v3.json'
LOCAL_ROOT = Path('/content/cottonlens-full-year')
# Immutable Drive packages restore onto local disk with checksums; no retraining.
BASE = [PYTHON,'-m','cottonlens_ml.research.engine','--repo',REPO,'--drive-root',LOCAL_ROOT,'--mirror-root',DRIVE_ROOT,'--experiment',EXPERIMENT,'--profile','full-year-v1']
if PROFILE != 'full-year-v1': BASE[-1] = PROFILE


if PROFILE in ('ams-exploration-v1','fas-exploration-v1','nass-exploration-v1','wasde-exploration-v1','cftc-exploration-v1','fx-exploration-v1','crop-exploration-v1','weather-exploration-v1','oncall-exploration-v1','oncall-exploration-v2') and WORKFLOW == 'prepare':
    # Frozen small input packet only; does not hydrate old model payloads.
    import shutil
    packet = DRIVE_ROOT/'data'/('oncall-exploration-v2-inputs-20261002' if PROFILE == 'oncall-exploration-v2' else 'oncall-exploration-inputs-20261002' if PROFILE == 'oncall-exploration-v1' else 'weather-exploration-inputs-20261002' if PROFILE == 'weather-exploration-v1' else 'crop-exploration-inputs-20261002' if PROFILE == 'crop-exploration-v1' else 'fx-exploration-inputs-20261002' if PROFILE == 'fx-exploration-v1' else 'cftc-exploration-inputs-20261002' if PROFILE == 'cftc-exploration-v1' else 'wasde-exploration-inputs-20261002' if PROFILE == 'wasde-exploration-v1' else 'nass-exploration-inputs-20261002' if PROFILE == 'nass-exploration-v1' else 'fas-exploration-inputs-20261002' if PROFILE == 'fas-exploration-v1' else 'ams-exploration-inputs-20261002')
    receipt = json.loads((packet/'input-manifest.json').read_text())
    local = Path('/content')/('cottonlens-oncall-v2-inputs' if PROFILE == 'oncall-exploration-v2' else 'cottonlens-oncall-inputs' if PROFILE == 'oncall-exploration-v1' else 'cottonlens-weather-inputs' if PROFILE == 'weather-exploration-v1' else 'cottonlens-crop-inputs' if PROFILE == 'crop-exploration-v1' else 'cottonlens-fx-inputs' if PROFILE == 'fx-exploration-v1' else 'cottonlens-cftc-inputs' if PROFILE == 'cftc-exploration-v1' else 'cottonlens-wasde-inputs' if PROFILE == 'wasde-exploration-v1' else 'cottonlens-nass-inputs' if PROFILE == 'nass-exploration-v1' else 'cottonlens-fas-inputs' if PROFILE == 'fas-exploration-v1' else 'cottonlens-ams-inputs')
    for name, checksum in receipt['files'].items():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts
        source, target = packet/name, local/name
        assert hashlib.sha256(source.read_bytes()).hexdigest() == checksum
        target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists(): shutil.copyfile(source,target)
        assert hashlib.sha256(target.read_bytes()).hexdigest() == checksum
    reference = local/'reference'
    if PROFILE in ('oncall-exploration-v1','oncall-exploration-v2'):
        BASE += ['--oncall-table',local/'oncall-2010-2023.csv']
    elif PROFILE == 'weather-exploration-v1':
        BASE += ['--weather-table',local/'weather-2010-2023.csv']
    elif PROFILE == 'crop-exploration-v1':
        BASE += ['--crop-table',local/'crop-prices-2010-2023.csv']
    elif PROFILE == 'fx-exploration-v1':
        BASE += ['--fx-table',local/'currency-crosses-2010-2023.csv']
    elif PROFILE == 'cftc-exploration-v1':
        BASE += ['--cftc-table',local/'positions-2010-2023.csv']
    elif PROFILE == 'wasde-exploration-v1':
        BASE += ['--wasde-table',local/'world-balance-2016-2023.csv']
    elif PROFILE == 'nass-exploration-v1':
        BASE += ['--nass-table',local/'upland-condition-2010-2023.csv','--nass-audit-root',local]
    elif PROFILE == 'fas-exploration-v1':
        BASE += ['--fas-table',local/'weekly-sales-2010-2023.csv']
    else:
        BASE += ['--ams-table',local/'spot-quotations-2020-2023.csv',
                 '--ams-publications',local/'publication-content-review.json']

if PROFILE in ('recency-pilot-v1','return-path-pilot-v1','agri-transfer-pilot-v1','agri-nonlinear-pilot-v1') and WORKFLOW in ('prepare', 'pilot'):
    # Only a frozen small packet; never restore old control model payloads.
    import shutil
    packet = DRIVE_ROOT/('data/agri-nonlinear-pilot-v1-inputs-20261003' if PROFILE == 'agri-nonlinear-pilot-v1' else 'data/agri-transfer-pilot-v1-inputs-20261003' if PROFILE == 'agri-transfer-pilot-v1' else 'data/return-path-pilot-v1-inputs-20261003' if PROFILE == 'return-path-pilot-v1' else 'data/recency-pilot-v1-inputs-20261003')
    receipt = json.loads((packet/'input-manifest.json').read_text())
    local = Path('/content/cottonlens-nonlinear-inputs' if PROFILE == 'agri-nonlinear-pilot-v1' else '/content/cottonlens-transfer-inputs' if PROFILE == 'agri-transfer-pilot-v1' else '/content/cottonlens-path-inputs' if PROFILE == 'return-path-pilot-v1' else '/content/cottonlens-recency-inputs')
    for name, checksum in receipt['files'].items():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts
        source, target = packet/name, local/name
        assert hashlib.sha256(source.read_bytes()).hexdigest() == checksum
        target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists(): shutil.copyfile(source,target)
        assert hashlib.sha256(target.read_bytes()).hexdigest() == checksum
    reference = local if PROFILE in ('agri-transfer-pilot-v1','agri-nonlinear-pilot-v1') else local/'reference'


In [ ]:
import colab_progress
from IPython.display import display, Markdown
if WORKFLOW == 'review':
    run([PYTHON,'-m','cottonlens_ml.research.review','--experiment-root',reference,'--evidence-root',EVIDENCE_ROOT,'--readiness',READINESS,'--output',DRIVE_ROOT/'reports','--extended'],env=cpu_env())
elif WORKFLOW == 'pilot':
    assert RUN_TRAINING, 'Select pilot and RUN_TRAINING=True deliberately'
    if PROFILE in ('recency-pilot-v1','return-path-pilot-v1','agri-transfer-pilot-v1','agri-nonlinear-pilot-v1'):
        run([*BASE,'--stage','prepare','--reference-root',reference],env=cpu_env())
    panel=display(Markdown('Starting bounded pilot; cache is separate from new fits.'),display_id=True)
    def update(message):
        panel.update(Markdown('```text\n'+message+'\n```'))
    colab_progress.supervise([*BASE,'--stage','pilot','--max-minutes',str(MAX_MINUTES)],cwd=REPO,env=cpu_env(),drive_logs=DRIVE_ROOT/'reports',decisions=LOCAL_ROOT/'experiments'/EXPERIMENT/('nonlinear-outputs' if PROFILE == 'agri-nonlinear-pilot-v1' else 'transfer-outputs' if PROFILE == 'agri-transfer-pilot-v1' else 'path-outputs' if PROFILE == 'return-path-pilot-v1' else 'recency-outputs' if PROFILE == 'recency-pilot-v1' else 'information-outputs' if PROFILE in ('ams-exploration-v1','fas-exploration-v1','nass-exploration-v1','wasde-exploration-v1','cftc-exploration-v1','fx-exploration-v1','crop-exploration-v1','weather-exploration-v1','oncall-exploration-v1','oncall-exploration-v2') else 'pilot-full-year'),total=32 if PROFILE in ('return-path-pilot-v1','agri-transfer-pilot-v1','agri-nonlinear-pilot-v1') else 35 if PROFILE=='wasde-exploration-v1' else 56 if PROFILE in ('fas-exploration-v1','nass-exploration-v1','cftc-exploration-v1','fx-exploration-v1','crop-exploration-v1','weather-exploration-v1','oncall-exploration-v1','oncall-exploration-v2') else 7 if PROFILE=='ams-exploration-v1' else 16,decision_pattern='*.json',fold_count=1 if PROFILE=='ams-exploration-v1' else 5 if PROFILE=='wasde-exploration-v1' else 8,update=update)
else:
    for stage,_ in STAGES:
        command=[*BASE,'--stage',stage]
        if stage=='prepare': command += ['--reference-root',reference]
        run(command,env=cpu_env())

if PROFILE in ('recency-pilot-v1','return-path-pilot-v1','agri-transfer-pilot-v1','agri-nonlinear-pilot-v1') and WORKFLOW == 'pilot':
    # Analyze completed predictions automatically; pending is a normal planned pause.
    run([*BASE,'--stage','compare'],env=cpu_env())


A planned pause is resumable with the same frozen source/environment. A writer lock requires owner inspection and confirmed stopped runtime, never blind deletion. The first prepare freezes source, dates, labels and six hypotheses. Missing or failed outputs remain pending. Compare produces historical research decisions; it does not import artifacts or launch a wider search. The canonical protocol is docs/MASTER_PLAN_20261001.md.
